# ⚡ SOTA DL Anchor Training Pipeline (End-to-End)

This notebook implements the complete training process for the Deep Learning (PyTorch Lightning LSTM/Transformer) sequence models. 
It acts as a primary SOTA sequence feature generator, outputting 0% leak-free out-of-fold predictions that are subsequently stacked into the main GBDT ensemble.

### Flow:
1. **Core Settings & MPS Fallback Activation**
2. **Config Load & OmegaConf Setup**
3. **Verified Kaggle Grandmaster Generalization Hyperparameters Setup**
4. **Chronos Embedding Extraction & Sequence Modeling**
5. **5-Fold Chronological Sequence Training (DLTrainer)**
6. **Anchor OOF Prediction Map & Submission Alignments**

In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path

from omegaconf import OmegaConf

# ====================================================
# Kaggle Env Initialization & Dynamic Directory Copy
# ====================================================
if Path("/kaggle/input").exists():
    print("Detected Kaggle execution environment. Installing required packages...")
    os.system("pip install pytorch-lightning omegaconf -q")
    print("✅ Packages successfully installed!")

    # 1. Search for directory containing src and conf inside /kaggle/input
    found_src = False
    for root, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs and "conf" in dirs:
            src_root = Path(root)
            print(f"✅ Found source root at: {src_root}")

            # Copy essential codebase directories directly to target execution dir ./
            for folder in ["src", "conf", "scripts"]:
                src_folder = src_root / folder
                dst_folder = Path(folder)
                if src_folder.exists():
                    print(f"  Mapping {src_folder} -> ./{folder}")
                    try:
                        shutil.copytree(src_folder, dst_folder, dirs_exist_ok=True)
                    except Exception as e:
                        print(f"  Copy warning for {folder}: {e}")

            # Copy datasets (including parquets/labels) to ./data if present
            src_data = src_root / "data"
            if src_data.exists():
                print(f"  Mapping {src_data} -> ./data")
                try:
                    shutil.copytree(src_data, Path("data"), dirs_exist_ok=True)
                except Exception as e:
                    print(f"  Data folder copy notice: {e}")

            found_src = True
            break

    # 2. Fallback: If zip file exists instead of uncompressed folder, unzip it
    if not found_src:
        for root, dirs, files in os.walk("/kaggle/input"):
            for file in files:
                if file.endswith(".zip"):
                    zip_path = Path(root) / file
                    print(f"Extracting backup zip {zip_path}...")
                    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                        zip_ref.extractall(".")
                    print("Extraction complete!")

# Activate Pytorch MPS fallback for Mac hardware acceleration stability
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Ensure project root is in sys.path
project_root = Path(os.getcwd()).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.dl_trainer import DLTrainer
from src.utils.logger import get_logger

logger = get_logger(__name__)
print("✅ Core SOTA PyTorch deep learning libraries successfully imported!")

In [ ]:
print("🔄 Loading configurations...")
config_path = project_root / "conf" / "config.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"Config file not found at {config_path}")

cfg = OmegaConf.load(config_path)

# Override configurations for robust sequence deep modeling
cfg.env = "local"
cfg.model.name = "dl_transformer"       # Deep sequential transformer backbone
cfg.model.seq_len = 14                 # 2-week history sequence length
cfg.model.epochs = 20                  # Adequate epochs on CPU
cfg.training.tune_hyperparameters = False

# High SOTA-generalization hyperparams
cfg.model.lr = 0.00549
cfg.model.hidden_dim = 256
cfg.model.num_layers = 3
cfg.model.dropout = 0.26
cfg.model.batch_size = 32

print(f"Loaded Config: Model={cfg.model.name}, Epochs={cfg.model.epochs}, batch_size={cfg.model.batch_size}")

In [ ]:
print("🔄 Preparing Chronos Embeddings (if missing)...")
hf_emb_path = project_root / "data" / "processed" / "hf_chronos_embeddings.csv"
if hf_emb_path.exists():
    print(f"✅ Found existing HF Chronos embeddings at {hf_emb_path}. Skipping extraction to save CPU.")
else:
    try:
        print("Extracting HF Chronos Embeddings for temporal signals...")
        # Dynamically import and trigger embedding scripts
        sys.path.append(str(project_root / "scripts"))
        import extract_hf_features
        extract_hf_features.main()
    except Exception as e:
        print(f"⚠️ Chronos features extraction skipped/unsupported: {e}. Proceeding with base features.")

In [ ]:
print("⚡ Starting 5-Fold Sequential Deep Learning Training Loop...")
trainer = DLTrainer(cfg)
avg_score = trainer.run()
print(f"🎉 PyTorch Lightning Sequence Model Training Complete! 5-Fold Avg Log-Loss: {avg_score:.5f}")

In [ ]:
import shutil

print("🔄 Mapping newly trained optimal Deep Learning weights to GBDT ensemble anchors...")
submission_dir = project_root / "submissions"

new_oofs = sorted(submission_dir.glob("oof_optuna_trial_*_score*.csv"), key=os.path.getmtime)
new_subs = sorted(submission_dir.glob("submission_optuna_trial_*_score*.csv"), key=os.path.getmtime)

if new_oofs and new_subs:
    latest_oof = new_oofs[-1]
    latest_sub = new_subs[-1]

    target_oof_path = submission_dir / "oof_dl_v10.csv"
    target_sub_path = submission_dir / "submission_dl_v10.csv"

    shutil.copy2(latest_oof, target_oof_path)
    shutil.copy2(latest_sub, target_sub_path)

    print(f"✅ Successfully aligned: {latest_oof.name} -> oof_dl_v10.csv")
    print(f"✅ Successfully aligned: {latest_sub.name} -> submission_dl_v10.csv")
else:
    alternative_oofs = sorted(submission_dir.glob("oof_*.csv"), key=os.path.getmtime)
    aligned = False
    for f in alternative_oofs:
        if "score" in f.name and "dl" in f.name:
            shutil.copy2(f, submission_dir / "oof_dl_v10.csv")
            print(f"✅ Aligned alternative OOF: {f.name} -> oof_dl_v10.csv")
            aligned = True
            break
    if not aligned:
        print("⚠️ No new OOF matching DL schema found to copy. Verify folder state.")